In [13]:
import pandas as pd
import numpy as np
import torch
import re
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

BASE       = '../../outputs/final'
ENTITY_PATH = f'{BASE}/entity_nodes.csv'
EDGES_PATH  = f'{BASE}/knowledge_edges.csv'
OUT_DIR     = '../../outputs/intermediate/'

MIN_DEGREE = 15
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: cpu


In [14]:
en = pd.read_csv(ENTITY_PATH)
ke = pd.read_csv(EDGES_PATH)
ke = ke[ke['year'] < 2022].dropna(subset=['year'])

degree = ke.groupby('target')['source'].nunique().reset_index().rename(columns={'source':'degree'})
high_ids = set(degree[degree['degree'] >= MIN_DEGREE]['target'])

GENERIC = {
    'results','model','models','performance','baseline','baselines','training',
    'better','best','accuracy','improvement','improvements','outperforms','outperform',
    'words','best performance','better performance','our model','ablation analysis',
    'hyperparameters','experimental setup','method','methods','higher','lower','well',
    'scores','score','similarly','comparable results','proposed model','both datasets',
    'sentence pairs','both tasks','most cases','layers','predictions','entities',
    'proposed approach','significantly improves','significantly improved','performance gain',
    'inference','document','sequence','reward','vocabulary size','seq2seq','regularization',
    'comparable','significantly','notably','recently','however','although','overall',
    'previous','following','thus','hence','since','while','both','each','several',
    'various','all','most','many','some','other','our','their','its','such',
    'slightly better', 'highest performance', 'our approach', 'without', 'advantages'
    'our system', 'our systems', 'all systems', 'all tasks', 'all baselines',
    'aligning', 'unlike', 'measuring', 'problem', 'our framework', 'each model',
    'cognitive', 'advantages', 'frequency', 'distinct', 'creating', 'targeted', 'neural', 'of deep learning',
    'recognising textual entailment', 'clinical', 'fluency', 'vocabulary'
}
SINGLE = {
    'results','model','training','accuracy','loss','layers','words','scores','method',
    'higher','lower','better','best','well','reward','inference','document','sequence',
    'entities','baseline','models','methods','performance','decoder','encoder','bleu',
    'dropout','dimension','questions','attention','embeddings','embedding','features',
    'dataset','datasets','task','tasks','output','outputs','input','inputs','weight',
    'weights','layer','network','networks','system','systems','approach','similar',
    'different','large','small','high','low','good','new','first','last','both','each',
    'more','less','rate','norm','image','benefit','adam','relevance','generator',
    'optimizer','dimensions','pytorch','tensorflow', 'worse', 'poorly', 'beneficial', 'gain', 'full model',
    'our framework', 'novel framework', 'performances', 'test set',
    'robust', 'relevant', 'existing', 'structural', 'scaling',
    'increases', 'integrated', 'integrative', 'interactive',
    'despite', 'vision', 'climate'
}

def is_clean(name):
    if not isinstance(name, str): return False
    n = name.strip().lower()
    words = n.split()
    if not (1 <= len(words) <= 4): return False
    if re.search(r'[,;()]', n): return False
    if n in GENERIC: return False
    if re.match(r'^[\-\[\]0-9\"\']', n): return False
    if words[-1] in {'the','a','an','of','in','for','to','and','or','with','from','by','as','is','are'}: return False
    if words[0] in {'the','a','an'}: return False
    if len(n) < 4: return False
    if len(words) == 1 and len(n) < 6: return False
    if len(words) == 1 and n in SINGLE: return False
    return True

en_clean = en[en['node_id'].isin(high_ids) & en['name'].apply(is_clean)].copy()
en_clean = en_clean.reset_index(drop=True)
print(f'Entities to embed: {len(en_clean):,}')
en_clean.to_csv(f'{OUT_DIR}/entity_clean_index.csv', index=False)

Entities to embed: 553


In [16]:
tokenizer = AutoTokenizer.from_pretrained('allenai/scibert_scivocab_uncased')
model_sci  = AutoModel.from_pretrained('allenai/scibert_scivocab_uncased')
model_sci.to(DEVICE)
model_sci.eval()
print('SciBERT loaded.')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 641.71it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical ar

SciBERT loaded.


In [17]:
def embed_texts(texts, batch_size=64):
    all_embs = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size)):
            batch = texts[i:i+batch_size]
            enc = tokenizer(batch, padding=True, truncation=True,
                            max_length=32, return_tensors='pt').to(DEVICE)
            out = model_sci(**enc)
            emb = out.last_hidden_state[:, 0, :].cpu().numpy()  # CLS token
            all_embs.append(emb)
    return np.vstack(all_embs)

names = en_clean['name'].tolist()
entity_embeddings = embed_texts(names)
print(f'Entity embeddings shape: {entity_embeddings.shape}')

100%|██████████| 9/9 [00:02<00:00,  4.18it/s]

Entity embeddings shape: (553, 768)


In [22]:
np.save('../../outputs/intermediate/entity_embeddings.npy', entity_embeddings)
np.save('../../outputs/intermediate/entity_ids.npy', np.array(en_clean['node_id'].tolist()))
print(f'Saved entity_embeddings.npy  {entity_embeddings.shape}')
print(f'Saved entity_ids.npy  ({len(en_clean)} ids)')

Saved entity_embeddings.npy  (553, 768)
Saved entity_ids.npy  (553 ids)
